# TUẦN 3 - BẢO: ResNet50 + FAISS

Notebook này được tách từ phần baseline tuần 2 và giữ lại các phần cần thiết:
- Import thư viện
- Đường dẫn dữ liệu
- Đọc `train.csv`
- Tạo `candidate_df` chung
- Tiền xử lý ảnh
- Load ResNet50
- Trích xuất `feature_matrix`
- So sánh **cosine brute-force** và **FAISS IndexFlatIP**
- Xuất bảng metric và bảng tốc độ

Ghi nhớ: **FAISS chỉ tăng tốc truy xuất, không tự làm mAP tăng.**


## CELL 1: Import thư viện


In [13]:
# Dữ liệu
import os
import time
import random
import numpy as np
import pandas as pd

# Ảnh & PyTorch
import torch
import torch.nn.functional as F
import torchvision.transforms as t
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# Tìm kiếm tương đồng
import faiss

# Vẽ biểu đồ / hiển thị
import matplotlib.pyplot as plt
from tqdm import tqdm

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
plt.rcParams['figure.figsize'] = (10, 5)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('Import thư viện thành công!')
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {"cuda" if torch.cuda.is_available() else "cpu"}')


Import thư viện thành công!
PyTorch version : 2.12.0+cpu
Device          : cpu


## CELL 2: Đường dẫn dữ liệu

Sửa `DATA_DIR` đúng với máy đang chạy. Đây là chỗ sai là cả notebook nằm thở oxy.


In [14]:
DATA_DIR  = r'd:\Study\docs\Python\workspace\DoAnPython\project\data\raw'

CSV_PATH  = os.path.join(DATA_DIR, 'train.csv')
IMAGE_DIR = os.path.join(DATA_DIR, 'train_images')

# Thư mục lưu kết quả
PROCESSED = '../data/processed/'
RESULTS   = '../results/'
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

# Kiểm tra file/thư mục
for p in [CSV_PATH, IMAGE_DIR]:
    status = 'Đã tìm thấy' if os.path.exists(p) else 'Không tìm thấy - kiểm tra lại đường dẫn!'
    print(f'{status}: {p}')


Đã tìm thấy: d:\Study\docs\Python\workspace\DoAnPython\project\data\raw\train.csv
Đã tìm thấy: d:\Study\docs\Python\workspace\DoAnPython\project\data\raw\train_images


## CELL 3: Đọc dataset


In [15]:
df = pd.read_csv(CSV_PATH)

print(f'Train: {df.shape[0]:,} dòng x {df.shape[1]:,} cột')
print(f'Các cột: {list(df.columns)}')
df.head()


Train: 34,250 dòng x 5 cột
Các cột: ['posting_id', 'image', 'image_phash', 'title', 'label_group']


,posting_id,image,image_phash,title,label_group
0,train_129225211,0000a68812bc7e98c42888dfb1c07da0.jpg,94974f937d4c2433,Paper Bag Victoria Secret,249114794
1,train_3386243561,00039780dfc94d01db8676fe789ecd05.jpg,af3f9460c2838f0f,"Double Tape 3M VHB 12 mm x 4,5 m ORIGINAL / DO...",2937985045
2,train_2288590299,000a190fdd715a2a36faed16e2c65df7.jpg,b94cb00ed3e50f78,Maling TTS Canned Pork Luncheon Meat 397 gr,2395904891
3,train_2406599165,00117e4fc239b1b641ff08340b429633.jpg,8514fc58eafea283,Daster Batik Lengan pendek - Motif Acak / Camp...,4093212188
4,train_3369186413,00136d1cf4edede0203f32f05f660588.jpg,a6f319f924ad708c,Nescafe \xc3\x89clair Latte 220ml,3648931069


## CELL 4: Tạo `candidate_df` chung để đánh giá

Phần này dựa trên tuần 2, nhưng chỉnh lại cho chắc: lấy theo **nhóm**, mỗi nhóm **2 ảnh**, để mỗi query có ít nhất 1 ảnh cùng `label_group` trong gallery. Không làm vậy thì metric nhìn như có vẻ thông minh, thật ra là đang tự lừa mình.


In [16]:
# Đếm số ảnh mỗi label_group
so_anh_moi_nhom = df['label_group'].value_counts()

# Chỉ giữ nhóm có từ 2 ảnh trở lên
nhom_hop_le = so_anh_moi_nhom[so_anh_moi_nhom >= 2].index
df_loc = df[df['label_group'].isin(nhom_hop_le)].copy()

# Chọn 250 nhóm, mỗi nhóm lấy 2 ảnh -> tổng 500 ảnh
EVAL_GROUPS = 250
VALID_GROUPS = df_loc['label_group'].drop_duplicates().sample(
    n=min(EVAL_GROUPS, df_loc['label_group'].nunique()),
    random_state=SEED
)

danh_sach = []
for nhom in VALID_GROUPS:
    du_lieu_nhom = df_loc[df_loc['label_group'] == nhom]
    danh_sach.append(du_lieu_nhom.sample(n=2, random_state=SEED))

candidate_df = pd.concat(danh_sach, ignore_index=True)
candidate_df = candidate_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Lưu để các baseline khác dùng lại cho công bằng
candidate_path = os.path.join(PROCESSED, 'candidate_df_tuan3.csv')
candidate_df.to_csv(candidate_path, index=False)

nhom_counts = candidate_df['label_group'].value_counts()
print(f'candidate_df: {len(candidate_df)} ảnh, {candidate_df["label_group"].nunique()} nhóm')
print(f'Nhóm có đúng 2 ảnh: {(nhom_counts == 2).sum()} nhóm')
print(f'Nhóm có 1 ảnh     : {(nhom_counts == 1).sum()} nhóm')
print(f'Đã lưu candidate_df vào: {candidate_path}')

candidate_df.head()


candidate_df: 500 ảnh, 250 nhóm
Nhóm có đúng 2 ảnh: 250 nhóm
Nhóm có 1 ảnh     : 0 nhóm
Đã lưu candidate_df vào: ../data/processed/candidate_df_tuan3.csv


,posting_id,image,image_phash,title,label_group
0,train_2335515391,48bd37700cb4d84a7a30e364505a9adc.jpg,b7b4856b19c15a9c,Happy Call Grill Pan Alat Pemanggang,3893786536
1,train_1403240912,9c95c4fc5d37a29af29f14c0e4d4596e.jpg,ae87d86c87b3c0b8,JONICAT perangkap tikus jebakan tikus JONI CAT...,1126912250
2,train_4292488993,0e2e9f8ea4e877e0482cf14f7a9de1c8.jpg,f916c1e19e1f0d4a,[READY STOCK] Follow Me Green Tea Shampoo Mala...,1151643989
3,train_400351908,0831db0b54ec0db05651ba29fd814826.jpg,df9175e4624a626a,Lip tint madame gie / liptint madame.gie ala k...,2966681171
4,train_3365654772,337a6857ba721ac3518fddcdba0bcc2f.jpg,e1329bcdcecd0c8c,1KG BISA 4PCS | BAGGY JEANS SAKU / POCKET RAWI...,4043587155


## CELL 5: Tiền xử lý ảnh


In [17]:
transform_pipeline = t.Compose([
    t.Resize((224, 224)),
    t.CenterCrop(224),
    t.ToTensor(),
    t.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def load_image(path):
    img = Image.open(path).convert('RGB')
    return transform_pipeline(img).unsqueeze(0)

# Kiểm tra nhanh 1 ảnh
sample_img = os.path.join(IMAGE_DIR, candidate_df['image'].iloc[0])
print(f'Ảnh kiểm tra: {sample_img}')

if os.path.exists(sample_img):
    tensor = load_image(sample_img)
    print(f'Kích thước tensor: {tensor.shape}')
else:
    print('Không tìm thấy ảnh kiểm tra - xem lại IMAGE_DIR.')


Ảnh kiểm tra: d:\Study\docs\Python\workspace\DoAnPython\project\data\raw\train_images\48bd37700cb4d84a7a30e364505a9adc.jpg
Kích thước tensor: torch.Size([1, 3, 224, 224])


## CELL 6: Load ResNet50 pretrained và bỏ lớp phân loại cuối


In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model.eval()
model = model.to(device)

# Lấy backbone, bỏ fully-connected layer cuối để trích vector 2048 chiều
feature_extractor = torch.nn.Sequential(*list(model.children())[:-1]).to(device)
feature_extractor.eval()

print(f'Load ResNet50 thành công. Sử dụng: {device}')


Load ResNet50 thành công. Sử dụng: cpu


## CELL 7: Dataset + DataLoader + trích xuất feature


In [19]:
class ImageDataset(Dataset):
    def __init__(self, image_list):
        self.image_list = image_list

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_name = self.image_list[idx]
        path = os.path.join(IMAGE_DIR, img_name)
        img_tensor = load_image(path).squeeze(0)
        return img_tensor, img_name

image_list = candidate_df['image'].tolist()
labels = candidate_df['label_group'].values

dataset = ImageDataset(image_list)
loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)

all_features = []
print('Bắt đầu trích xuất feature ResNet50...')

with torch.no_grad():
    for imgs, fnames in tqdm(loader):
        imgs = imgs.to(device)
        feats = feature_extractor(imgs)
        feats = feats.view(feats.size(0), -1).cpu()
        all_features.append(feats)

feature_matrix = torch.cat(all_features, dim=0)
print(f'feature_matrix: {feature_matrix.shape}')

# Lưu feature để lần sau khỏi trích xuất lại
feature_path = os.path.join(PROCESSED, 'resnet50_features_tuan3.npy')
np.save(feature_path, feature_matrix.numpy())
print(f'Đã lưu feature vào: {feature_path}')


Bắt đầu trích xuất feature ResNet50...


100%|██████████| 16/16 [01:42<00:00,  6.39s/it]

feature_matrix: torch.Size([500, 2048])
Đã lưu feature vào: ../data/processed/resnet50_features_tuan3.npy


## CELL 8: Hàm tính Precision@K, Recall@K, mAP

Giữ công thức cùng hướng với tuần 2 để so sánh công bằng.


In [20]:
K_LIST = [1, 3, 5, 10]
MAX_K = max(K_LIST)

def average_precision(ranked_labels, true_label, total_relevant):
    """Tính AP cho 1 query."""
    if total_relevant == 0:
        return 0.0

    ap = 0.0
    hits = 0
    for rank, label in enumerate(ranked_labels, 1):
        if label == true_label:
            hits += 1
            ap += hits / rank

    return ap / total_relevant


def evaluate_from_indices(top_indices, labels, method_name):
    """
    top_indices: array/list shape (N, MAX_K), mỗi dòng là index ảnh truy xuất cho query i.
    labels: label_group tương ứng candidate_df.
    """
    rows = []
    all_ap = []

    for i in range(len(labels)):
        true_label = labels[i]
        total_relevant = sum(1 for j in range(len(labels)) if labels[j] == true_label and j != i)

        if total_relevant == 0:
            continue

        ranked_idx = list(top_indices[i])[:MAX_K]
        ranked_labels = [labels[j] for j in ranked_idx]

        ap = average_precision(ranked_labels, true_label, total_relevant)
        all_ap.append(ap)

        row = {
            'method': method_name,
            'image': candidate_df['image'].iloc[i],
            'label_group': true_label,
            'AP': round(ap, 4),
        }

        for k in K_LIST:
            top_k_labels = ranked_labels[:k]
            hits = sum(1 for lbl in top_k_labels if lbl == true_label)
            row[f'Precision@{k}'] = round(hits / k, 4)
            row[f'Recall@{k}'] = round(hits / min(total_relevant, k), 4)

        rows.append(row)

    detail_df = pd.DataFrame(rows)

    summary = []
    for k in K_LIST:
        summary.append({
            'method': method_name,
            'K': k,
            'Precision@K': round(detail_df[f'Precision@{k}'].mean(), 4),
            'Recall@K': round(detail_df[f'Recall@{k}'].mean(), 4),
            'mAP': round(float(np.mean(all_ap)), 4)
        })

    metrics_df = pd.DataFrame(summary)
    return metrics_df, detail_df


## CELL 9: Baseline cũ - Cosine brute-force

Đây là cách tuần 2: tính toàn bộ ma trận similarity `N x N`. Dễ hiểu, nhưng khi dữ liệu lớn thì chậm và tốn RAM, rất hợp với truyền thống làm máy tính đau khổ.


In [21]:
# Chuẩn hóa L2 để dot product = cosine similarity
features_norm = F.normalize(feature_matrix, p=2, dim=1)
features_np = features_norm.numpy().astype('float32')

start = time.perf_counter()

# Brute-force cosine: nhân ma trận N x N
sim_matrix = features_np @ features_np.T
np.fill_diagonal(sim_matrix, -1.0)

# Lấy top MAX_K index theo điểm similarity giảm dần
brute_top_indices = np.argsort(-sim_matrix, axis=1)[:, :MAX_K]

brute_time = time.perf_counter() - start

brute_metrics_df, brute_detail_df = evaluate_from_indices(
    brute_top_indices,
    labels,
    method_name='ResNet50 + Cosine brute-force'
)

print(f'Thời gian brute-force: {brute_time:.6f} giây')
print(brute_metrics_df.to_string(index=False))


Thời gian brute-force: 0.134254 giây
                       method  K  Precision@K  Recall@K   mAP
ResNet50 + Cosine brute-force  1       0.6780     0.678 0.727
ResNet50 + Cosine brute-force  3       0.2526     0.758 0.727
ResNet50 + Cosine brute-force  5       0.1572     0.786 0.727
ResNet50 + Cosine brute-force 10       0.0842     0.842 0.727


## CELL 10: FAISS IndexFlatIP

`IndexFlatIP` dùng inner product. Vì vector đã L2-normalize nên inner product chính là cosine similarity.


In [22]:
start = time.perf_counter()

# Tạo FAISS index
d = features_np.shape[1]
index = faiss.IndexFlatIP(d)
index.add(features_np)

# Search MAX_K + 1 vì kết quả đầu thường là chính ảnh query
scores, indices = index.search(features_np, MAX_K + 1)

# Bỏ chính nó khỏi danh sách kết quả
faiss_top_indices = []
for query_idx, row in enumerate(indices):
    filtered = [idx for idx in row if idx != query_idx]
    faiss_top_indices.append(filtered[:MAX_K])
faiss_top_indices = np.array(faiss_top_indices)

faiss_time = time.perf_counter() - start

faiss_metrics_df, faiss_detail_df = evaluate_from_indices(
    faiss_top_indices,
    labels,
    method_name='ResNet50 + FAISS IndexFlatIP'
)

print(f'Thời gian FAISS: {faiss_time:.6f} giây')
print(faiss_metrics_df.to_string(index=False))


Thời gian FAISS: 0.094378 giây
                      method  K  Precision@K  Recall@K   mAP
ResNet50 + FAISS IndexFlatIP  1       0.6780     0.678 0.727
ResNet50 + FAISS IndexFlatIP  3       0.2526     0.758 0.727
ResNet50 + FAISS IndexFlatIP  5       0.1572     0.786 0.727
ResNet50 + FAISS IndexFlatIP 10       0.0842     0.842 0.727


## CELL 11: So sánh tốc độ và metric


In [23]:
metrics_compare_df = pd.concat([brute_metrics_df, faiss_metrics_df], ignore_index=True)

speedup = brute_time / faiss_time if faiss_time > 0 else np.nan
speed_compare_df = pd.DataFrame([
    {
        'method': 'ResNet50 + Cosine brute-force',
        'time_seconds': round(brute_time, 6),
        'speedup_vs_bruteforce': 1.0
    },
    {
        'method': 'ResNet50 + FAISS IndexFlatIP',
        'time_seconds': round(faiss_time, 6),
        'speedup_vs_bruteforce': round(speedup, 2)
    }
])

print('BẢNG SO SÁNH METRIC')
print(metrics_compare_df.to_string(index=False))

print('\nBẢNG SO SÁNH TỐC ĐỘ')
print(speed_compare_df.to_string(index=False))

print('\nKết luận ngắn:')
print('- FAISS giúp tăng tốc truy xuất khi dữ liệu lớn hơn.')
print('- Với IndexFlatIP, FAISS là exact search nên metric gần như giữ nguyên so với cosine brute-force.')
print('- FAISS không tự làm mAP tăng. Muốn tăng mAP cần đặc trưng tốt hơn: CLIP, image+text, fine-tuning, hoặc metric learning.')


BẢNG SO SÁNH METRIC
                       method  K  Precision@K  Recall@K   mAP
ResNet50 + Cosine brute-force  1       0.6780     0.678 0.727
ResNet50 + Cosine brute-force  3       0.2526     0.758 0.727
ResNet50 + Cosine brute-force  5       0.1572     0.786 0.727
ResNet50 + Cosine brute-force 10       0.0842     0.842 0.727
 ResNet50 + FAISS IndexFlatIP  1       0.6780     0.678 0.727
 ResNet50 + FAISS IndexFlatIP  3       0.2526     0.758 0.727
 ResNet50 + FAISS IndexFlatIP  5       0.1572     0.786 0.727
 ResNet50 + FAISS IndexFlatIP 10       0.0842     0.842 0.727

BẢNG SO SÁNH TỐC ĐỘ
                       method  time_seconds  speedup_vs_bruteforce
ResNet50 + Cosine brute-force      0.134254                   1.00
 ResNet50 + FAISS IndexFlatIP      0.094378                   1.42

Kết luận ngắn:
- FAISS giúp tăng tốc truy xuất khi dữ liệu lớn hơn.
- Với IndexFlatIP, FAISS là exact search nên metric gần như giữ nguyên so với cosine brute-force.
- FAISS không tự làm mAP tăng. Mu

## CELL 12: Lưu kết quả ra file CSV


In [24]:
metrics_path = os.path.join(RESULTS, 'tuan3_bao_faiss_metrics_compare.csv')
speed_path = os.path.join(RESULTS, 'tuan3_bao_faiss_speed_compare.csv')
brute_detail_path = os.path.join(RESULTS, 'tuan3_bao_bruteforce_detail.csv')
faiss_detail_path = os.path.join(RESULTS, 'tuan3_bao_faiss_detail.csv')

metrics_compare_df.to_csv(metrics_path, index=False)
speed_compare_df.to_csv(speed_path, index=False)
brute_detail_df.to_csv(brute_detail_path, index=False)
faiss_detail_df.to_csv(faiss_detail_path, index=False)

print('Đã lưu:')
print(metrics_path)
print(speed_path)
print(brute_detail_path)
print(faiss_detail_path)


Đã lưu:
../results/tuan3_bao_faiss_metrics_compare.csv
../results/tuan3_bao_faiss_speed_compare.csv
../results/tuan3_bao_bruteforce_detail.csv
../results/tuan3_bao_faiss_detail.csv


## CELL 13: Đoạn nhận xét để đưa vào báo cáo

Có thể copy đoạn dưới vào mục FAISS của báo cáo tuần 3.


In [25]:
print(f"""
Trong tuần 3, nhóm thay phương pháp cosine brute-force bằng FAISS IndexFlatIP để tăng tốc truy xuất ảnh tương đồng.
Trước khi đưa vào FAISS, vector đặc trưng ResNet50 được chuẩn hóa L2, do đó phép inner product trong IndexFlatIP tương đương cosine similarity.
Kết quả cho thấy FAISS giúp rút ngắn thời gian truy xuất so với brute-force, đặc biệt có ý nghĩa khi mở rộng dữ liệu lên toàn bộ tập ảnh.
Tuy nhiên, do IndexFlatIP là exact search và sử dụng cùng vector đặc trưng ResNet50, các chỉ số Precision@K, Recall@K và mAP gần như không thay đổi.
Điều này cho thấy FAISS chủ yếu giải quyết vấn đề tốc độ truy xuất, không tự cải thiện chất lượng xếp hạng.
Để cải thiện mAP, nhóm cần thử đặc trưng mạnh hơn như CLIP, kết hợp image + text hoặc fine-tuning mô hình trên dữ liệu thương mại điện tử.

Thời gian brute-force: {brute_time:.6f} giây
Thời gian FAISS      : {faiss_time:.6f} giây
Tốc độ cải thiện    : {speedup:.2f} lần
""")



Trong tuần 3, nhóm thay phương pháp cosine brute-force bằng FAISS IndexFlatIP để tăng tốc truy xuất ảnh tương đồng.
Trước khi đưa vào FAISS, vector đặc trưng ResNet50 được chuẩn hóa L2, do đó phép inner product trong IndexFlatIP tương đương cosine similarity.
Kết quả cho thấy FAISS giúp rút ngắn thời gian truy xuất so với brute-force, đặc biệt có ý nghĩa khi mở rộng dữ liệu lên toàn bộ tập ảnh.
Tuy nhiên, do IndexFlatIP là exact search và sử dụng cùng vector đặc trưng ResNet50, các chỉ số Precision@K, Recall@K và mAP gần như không thay đổi.
Điều này cho thấy FAISS chủ yếu giải quyết vấn đề tốc độ truy xuất, không tự cải thiện chất lượng xếp hạng.
Để cải thiện mAP, nhóm cần thử đặc trưng mạnh hơn như CLIP, kết hợp image + text hoặc fine-tuning mô hình trên dữ liệu thương mại điện tử.

Thời gian brute-force: 0.134254 giây
Thời gian FAISS      : 0.094378 giây
Tốc độ cải thiện    : 1.42 lần

